# Full Planner — Best Strategy Simulation Notebook

This notebook is dedicated to simulating, evaluating, and discovering the **best overall strategy** for SpaceInvasion Classic across full-empire progression (economy scaling, energy optimization, research unlocks, fleet composition, raiding, colonization, and action scheduling).

## 0. Imports

All Python standard-library and `spaceinvasion` domain imports used across the simulation notebook.

In [ ]:
import contextlib
import datetime
import json
import math
from pathlib import Path
import shutil
import sqlite3
import statistics
import sys
import tempfile
from typing import Any

import pandas as pd

for _cand in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_cand / "spaceinvasion" / "__init__.py").is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

from spaceinvasion import game_data
from spaceinvasion.action_graph import build_action_graph, merge_strategy_config_overrides
from spaceinvasion.config import StrategyConfig
from spaceinvasion.fleet import (
    DEFENSE_SLUG_BY_KEY,
    SHIP_SLUG_BY_KEY,
    _parse_coords_distance,
)
from spaceinvasion.logging_utils import redact_sensitive
from spaceinvasion.optimizer import encode_empire_state, run_optimizer, simulate_action_sequence
from spaceinvasion.ranking import (
    POINTS_PER_RESOURCE_DIVISOR,
    attach_action_etas,
    compute_action_eta,
    estimate_account_points,
    estimate_hourly_economy,
)
from spaceinvasion.state import EnergyBalance, GameState, PlanetState, Resources
from spaceinvasion.storage import GameDatabase
from spaceinvasion.strategy import PlannedAction, plan_actions

## 1. Utils

Shared path-resolution (`resolve_repo_root`, `resolve_active_data_root`), formatting (`fmt_duration`, `render_table`), JSON parsing (`_safe_json_dict`), statistical (`compute_mean_and_std`), and flight-physics (`estimate_physics_one_way_seconds`) utilities reused across all notebook sections.

In [ ]:
pd.set_option("display.max_columns", 28)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 64)


def resolve_repo_root() -> Path:
    """Locate repository root from any working directory without bare `__file__`."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "spaceinvasion" / "__init__.py").is_file():
            return cand
    for entry in list(sys.path):
        if not entry:
            continue
        p = Path(entry).resolve()
        if (p / "spaceinvasion" / "__init__.py").is_file():
            return p
    return Path(game_data.__file__).resolve().parents[1]


def resolve_active_data_root(repo_root: Path) -> Path:
    """Prefer isolated `cwd` data/state fixtures when present; otherwise use `repo_root`."""
    cwd = Path.cwd().resolve()
    if (cwd / "state" / "spaceinvasion.db").is_file() or (
        cwd / "data" / "attack_history.json"
    ).is_file():
        return cwd
    return repo_root


def fmt_duration(seconds: float) -> str:
    """Format seconds into readable `Xd Yh`, `Xh Ym`, or `Xm Ys` strings."""
    if not math.isfinite(seconds) or seconds <= 0.0:
        return "0s"
    sec_i = int(round(seconds))
    if sec_i < 60:
        return f"{sec_i}s"
    mins, rem_s = divmod(sec_i, 60)
    if mins < 60:
        return f"{mins}m {rem_s:02d}s"
    hrs, rem_m = divmod(mins, 60)
    if hrs < 48:
        return f"{hrs}h {rem_m:02d}m"
    days, rem_h = divmod(hrs, 24)
    return f"{days}d {rem_h:02d}h"


def render_table(headers: list[str], rows: list[list[Any]]) -> str:
    """Render an aligned ASCII grid."""
    str_rows = [[str(c) for c in row] for row in rows]
    widths = [len(h) for h in headers]
    for row in str_rows:
        for idx, cell in enumerate(row):
            if idx < len(widths):
                widths[idx] = max(widths[idx], len(cell))
    sep = "+-" + "-+-".join("-" * w for w in widths) + "-+"
    head = "| " + " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers)) + " |"
    body = [
        "| " + " | ".join((row[i] if i < len(row) else "").ljust(widths[i]) for i in range(len(widths))) + " |"
        for row in str_rows
    ]
    return "\n".join([sep, head, sep, *body, sep])


def _safe_json_dict(raw: Any) -> dict[str, Any]:
    if isinstance(raw, dict):
        return dict(raw)
    if isinstance(raw, str) and raw.strip():
        try:
            parsed = json.loads(raw)
            return dict(parsed) if isinstance(parsed, dict) else {}
        except ValueError:
            return {}
    return {}


def compute_mean_and_std(values: list[float]) -> tuple[float, float, float]:
    """Return `(mean, sample_std, population_std)` safely for any list length."""
    clean = [float(v) for v in values if math.isfinite(float(v))]
    if not clean:
        return 0.0, 0.0, 0.0
    mean_val = float(statistics.fmean(clean))
    sample_std = float(statistics.stdev(clean)) if len(clean) >= 2 else 0.0
    pop_std = float(statistics.pstdev(clean)) if len(clean) >= 1 else 0.0
    return mean_val, sample_std, pop_std


def estimate_physics_one_way_seconds(
    origin_coords: str,
    target_coords: str,
    *,
    min_ship_speed: float = 12_000.0,
    fleet_speed_factor: float = 5.0,
) -> float:
    """SpaceInvasion Classic one-way flight time formula in seconds."""
    dist = max(1.0, float(_parse_coords_distance(origin_coords, target_coords)))
    spd = max(100.0, float(min_ship_speed))
    fac = max(100.0 if fleet_speed_factor > 100.0 else max(1.0, float(fleet_speed_factor)), 1.0)
    return float(round(10.0 + (3500.0 / fac) * math.sqrt((10.0 * dist) / spd)))


def format_prerequisites(requirements: list[dict[str, Any]] | tuple[dict[str, Any], ...]) -> str:
    """Format tech-tree requirement dicts into compact `b7 L3, r2 L1` notation."""
    if not requirements:
        return "None"
    parts: list[str] = []
    for req in requirements:
        cat = str(req.get("category") or "")
        slug = str(req.get("slug") or "")
        lvl = int(req.get("level") or 0)
        if cat == "buildings":
            b_info = game_data.BUILDINGS.get(slug) or {}
            bid = b_info.get("id")
            code = f"b{bid}" if bid is not None else slug
        elif cat == "research":
            r_info = game_data.RESEARCHES.get(slug) or {}
            rid = r_info.get("id")
            code = str(rid) if rid else slug
        else:
            code = slug or str(req.get("name") or "?")
        parts.append(f"{code} L{lvl} ({req.get('name', slug)})")
    return ", ".join(parts)


def build_monotonic_base_times(levels_map: dict[int, dict[str, Any]], max_level: int) -> dict[int, float]:
    """Extract level-by-level `base_time_s`, repairing high-level wiki scientific-notation mantissa drops."""
    out: dict[int, float] = {}
    prev_time = 0.0
    prev_cost_sp = 0.0
    for lvl in range(1, max_level + 1):
        row = levels_map.get(lvl) or {}
        raw_t = float(row.get("base_time_s") or 0.0)
        cost_sp = float(game_data.spice_equivalent(row.get("cost") or {}))
        if lvl > 1 and prev_time > 0.0 and raw_t < prev_time:
            ratio = (cost_sp / prev_cost_sp) if prev_cost_sp > 0.0 else 1.51
            fixed_t = prev_time * max(1.05, ratio)
        else:
            fixed_t = max(0.0, raw_t)
        out[lvl] = fixed_t
        prev_time = fixed_t
        prev_cost_sp = cost_sp
    return out


def solar_cell_energy_per_unit(temperature_c: float) -> float:
    """Orbital Solar cell (`s9`) base energy output (`-160..-80C: 10`, `0C: 30`, `+40C: 40`, `+200C: 80`)."""
    clamped_t = max(-80.0, min(200.0, float(temperature_c)))
    return float(max(10.0, min(80.0, 30.0 + 0.25 * clamped_t)))


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
ACTIVE_DATA_ROOT = resolve_active_data_root(REPO_ROOT)
LIVE_DB_PATH = ACTIVE_DATA_ROOT / "state" / "spaceinvasion.db"
ATTACK_HISTORY_PATH = ACTIVE_DATA_ROOT / "data" / "attack_history.json"
RESOURCE_KEYS: tuple[str, ...] = ("pig_iron", "metal", "kryptonite", "spice")

print(f"Utils initialized (REPO_ROOT={REPO_ROOT}, ACTIVE_DATA_ROOT={ACTIVE_DATA_ROOT})")

## 2. MER-Aligned Read-Only SQLite Telemetry Queries

Following [docs/database-mer.md](../docs/database-mer.md), this cell copies `state/spaceinvasion.db` (+ `-wal` / `-shm` sidecars) into a temporary directory and executes **four relational SQL queries** leveraging the exact keys, soft-flags, and secondary indexes of the 22-table schema:

1. **Colony & Fleet-Slot Capacity (`PLANET_SNAPSHOTS` via `idx_planet_snapshots_pid_ts`)**: Extracts latest colony coordinates, `Fleet base` (`b10`) level (`1 + b10` concurrent flight slots), and engine research levels (`r4`, `r5`, `r6`).
2. **Resolved Combat & Target Classification Join (`BATTLE_REPORTS` $\bowtie$ `GAME_MESSAGES` on `msg_uid` $\bowtie$ `SPY_TARGETS` on `coords`)**: Joins 4-resource plunder, combat fleet losses (`attacker_lost_spice`), and spawned orbital rubble (`debris_spice_eq`) with target status (`is_spacia`, `is_inactive`, `status_color`, `bash_hits`), filtering out soft-merged/own rows (`is_own_planet = 0 AND merged_into = ''`).
3. **Dispatch-to-Origin Colony Join (`RAID_HISTORY` via `idx_raid_history_coords_ts` $\bowtie$ `ACTION_HISTORY` via `idx_action_history_ts`)**: Resolves the exact launching colony `origin_coords` for each dispatched raid rather than assuming a single static origin.
4. **Orbital Rubble & Bash Cooldown Context (`RECYCLE_REPORTS` via `idx_recycle_reports_coords_ts` & `BASH_BLOCKS` via `idx_bash_blocks_coords_ts`)**: Queries supplementary recycler harvests and active server bash blocks.

In [ ]:
SQL_LATEST_COLONY_SNAPSHOT = """
SELECT coords, temperature, fields_used, fields_max, buildings_json, researches_json
FROM planet_snapshots
ORDER BY ts DESC, id DESC
LIMIT 1
"""

SQL_LATEST_GAME_STATE_META = """
SELECT value
FROM meta
WHERE key = 'latest_game_state'
LIMIT 1
"""

SQL_BATTLE_REPORTS_MER_JOIN = """
SELECT
    br.id,
    br.ts,
    br.msg_uid,
    br.battle_id,
    br.msg_date_text,
    br.target_coords,
    br.outcome,
    br.rounds,
    br.pig_iron,
    br.metal,
    br.kryptonite,
    br.spice,
    br.loot_spice_eq,
    br.attacker_lost_spice,
    br.defender_lost_spice,
    br.debris_spice_eq,
    gm.ingested_ts AS msg_ingested_ts,
    gm.deleted_in_game,
    COALESCE(st.status_color, 'unknown') AS target_status_color,
    COALESCE(st.is_inactive, 0) AS target_is_inactive,
    COALESCE(st.is_spacia, 0) AS target_is_spacia,
    COALESCE(st.bash_hits, 0) AS target_bash_hits
FROM battle_reports br
LEFT JOIN game_messages gm ON gm.msg_uid = br.msg_uid
LEFT JOIN spy_targets st
       ON st.coords = br.target_coords
      AND st.is_own_planet = 0
      AND st.merged_into = ''
ORDER BY br.msg_date_text ASC, br.id ASC
"""

SQL_RAID_DISPATCH_ORIGIN_JOIN = """
SELECT
    rh.id,
    rh.target_coords,
    rh.attacked_ts,
    rh.is_exempt,
    rh.outcome,
    rh.loot_json,
    COALESCE(
        (
            SELECT ah.planet_coords
            FROM action_history ah
            WHERE ah.kind = 'ATTACK'
              AND ah.executed = 1
              AND ah.dry_run = 0
              AND ah.planet_coords IS NOT NULL
              AND ah.planet_coords != ''
              AND ABS(ah.ts - rh.attacked_ts) <= 5.0
            ORDER BY ABS(ah.ts - rh.attacked_ts) ASC
            LIMIT 1
        ),
        ?
    ) AS origin_coords
FROM raid_history rh
ORDER BY rh.attacked_ts ASC, rh.id ASC
"""

SQL_RECYCLE_HARVESTS = """
SELECT
    coords,
    recycler_count,
    harvested_pig_iron,
    harvested_metal,
    harvested_kryptonite,
    harvested_spice,
    harvested_spice_eq,
    ts
FROM recycle_reports
ORDER BY ts ASC, id ASC
"""

SQL_ACTIVE_BASH_BLOCKS = """
SELECT COUNT(DISTINCT target_coords)
FROM bash_blocks
WHERE blocked_until_ts > ?
"""


def load_attack_telemetry_readonly(
    db_path: Path,
    json_path: Path,
) -> dict[str, Any]:
    """Execute static MER SQL queries on an isolated temp copy of `spaceinvasion.db`."""
    origin_coords = "02:380:12"
    colony_temperature = 31
    colony_buildings: dict[str, int] = {}
    colony_researches: dict[str, int] = {}
    universe_speedup_factor = 5.0
    fleet_base_level = 0
    engine_tech_levels: dict[str, int] = {"r4": 0, "r5": 0, "r6": 0}
    battle_rows: list[dict[str, Any]] = []
    raid_rows: list[dict[str, Any]] = []
    recycle_rows: list[dict[str, Any]] = []
    active_bash_blocks: int = 0
    source_label = "cold_start"

    if db_path.is_file() and db_path.stat().st_size >= 100:
        with tempfile.TemporaryDirectory(prefix="si_fp_ro_") as tmp_dir:
            tmp_root = Path(tmp_dir)
            tmp_db = tmp_root / db_path.name
            shutil.copy2(db_path, tmp_db)
            tmp_db.chmod(0o600)
            for suffix in ("-wal", "-shm", "-journal"):
                sidecar = Path(f"{db_path}{suffix}")
                if sidecar.is_file():
                    tmp_sidecar = Path(f"{tmp_db}{suffix}")
                    shutil.copy2(sidecar, tmp_sidecar)
                    tmp_sidecar.chmod(0o600)

            ro_db = GameDatabase(
                tmp_db,
                attack_history_path=tmp_root / "ro_mirror.json",
                backup_on_upgrade=False,
            )
            try:
                conn = ro_db._conn
                ps = conn.execute(SQL_LATEST_COLONY_SNAPSHOT).fetchone()
                if ps:
                    if ps["coords"]:
                        origin_coords = str(ps["coords"])
                    if ps["temperature"] is not None:
                        colony_temperature = int(ps["temperature"])
                    bld_d = _safe_json_dict(ps["buildings_json"])
                    res_d = _safe_json_dict(ps["researches_json"])
                    colony_buildings = {
                        str(k): max(0, int(v or 0))
                        for k, v in bld_d.items()
                        if isinstance(v, (int, float))
                    }
                    colony_researches = {
                        str(k): max(0, int(v or 0))
                        for k, v in res_d.items()
                        if isinstance(v, (int, float))
                    }
                    fleet_base_level = max(0, int(colony_buildings.get("b10") or 0))
                    for rk in ("r4", "r5", "r6"):
                        engine_tech_levels[rk] = max(0, int(colony_researches.get(rk) or 0))

                meta_row = conn.execute(SQL_LATEST_GAME_STATE_META).fetchone()
                if meta_row and meta_row[0]:
                    gs_dict = _safe_json_dict(meta_row[0])
                    raw_spd = gs_dict.get("speedup_factor")
                    if isinstance(raw_spd, (int, float)) and math.isfinite(float(raw_spd)) and float(raw_spd) > 0.0:
                        universe_speedup_factor = float(raw_spd)

                battle_rows = [dict(r) for r in conn.execute(SQL_BATTLE_REPORTS_MER_JOIN).fetchall()]
                raid_rows = [
                    dict(r)
                    for r in conn.execute(SQL_RAID_DISPATCH_ORIGIN_JOIN, (origin_coords,)).fetchall()
                ]
                recycle_rows = [dict(r) for r in conn.execute(SQL_RECYCLE_HARVESTS).fetchall()]

                ref_ts = max(
                    [0.0]
                    + [float(r.get("attacked_ts") or 0.0) for r in raid_rows]
                    + [float(b.get("ts") or 0.0) for b in battle_rows]
                )
                active_bash_blocks = int(
                    conn.execute(SQL_ACTIVE_BASH_BLOCKS, (ref_ts,)).fetchone()[0]
                )
            finally:
                ro_db.close()

            if battle_rows or raid_rows:
                source_label = "sqlite:mer_joined(battle_reports+raid_history+spy_targets)"

    if not battle_rows and not raid_rows and json_path.is_file():
        raw_obj = _safe_json_dict(json_path.read_text(encoding="utf-8"))
        for row in raw_obj.get("history") or []:
            if isinstance(row, dict):
                raid_rows.append({**dict(row), "origin_coords": origin_coords})
        if raid_rows:
            source_label = "json:attack_history"

    dispatch_timestamps = sorted(
        float(r["attacked_ts"])
        for r in raid_rows
        if isinstance(r.get("attacked_ts"), (int, float)) and math.isfinite(float(r["attacked_ts"]))
    )

    tz_offsets_sec: list[float] = []
    parsed_battle_naive: list[float | None] = []
    for brow in battle_rows:
        dt_txt = str(brow.get("msg_date_text") or "").strip()
        try:
            naive_ts = (
                datetime.datetime.strptime(dt_txt, "%Y-%m-%d %H:%M:%S")
                .replace(tzinfo=datetime.timezone.utc)
                .timestamp()
            )
        except ValueError:
            naive_ts = None
        parsed_battle_naive.append(naive_ts)
        if naive_ts is not None:
            for rrow in raid_rows:
                if str(rrow.get("target_coords") or "") != str(brow.get("target_coords") or ""):
                    continue
                rts = float(rrow.get("attacked_ts") or 0.0)
                cand_h = round((rts - naive_ts) / 3600.0)
                if abs(cand_h) <= 14 and 30.0 <= (naive_ts + cand_h * 3600.0 - rts) <= 3600.0:
                    tz_offsets_sec.append(cand_h * 3600.0)

    server_tz_offset_sec = statistics.median(tz_offsets_sec) if tz_offsets_sec else 0.0

    records: list[dict[str, Any]] = []
    used_raid_ids: set[int] = set()
    if battle_rows:
        matched_pairs_sec_per_sqrtdist: list[float] = []
        provisional_rows: list[tuple[dict[str, Any], float, float | None, str]] = []
        for brow, naive_ts in zip(battle_rows, parsed_battle_naive):
            t_coords = str(brow.get("target_coords") or "")
            impact_ts = (
                (naive_ts + server_tz_offset_sec)
                if naive_ts is not None
                else float(brow.get("ts") or 0.0)
            )
            matched_one_way: float | None = None
            matched_origin = origin_coords
            best_rid: int | None = None
            for idx_r, rrow in enumerate(raid_rows):
                if idx_r in used_raid_ids or str(rrow.get("target_coords") or "") != t_coords:
                    continue
                rts = float(rrow.get("attacked_ts") or 0.0)
                dt_flight = impact_ts - rts
                if 30.0 <= dt_flight <= 7200.0:
                    if matched_one_way is None or dt_flight < matched_one_way:
                        matched_one_way = dt_flight
                        matched_origin = str(rrow.get("origin_coords") or origin_coords)
                        best_rid = idx_r
            if best_rid is not None and matched_one_way is not None:
                used_raid_ids.add(best_rid)
                dist = max(1.0, float(_parse_coords_distance(matched_origin, t_coords)))
                matched_pairs_sec_per_sqrtdist.append(max(1.0, matched_one_way - 10.0) / math.sqrt(dist))
            provisional_rows.append((brow, impact_ts, matched_one_way, matched_origin))

        emp_scale = (
            statistics.median(matched_pairs_sec_per_sqrtdist)
            if matched_pairs_sec_per_sqrtdist
            else None
        )
        for brow, impact_ts, matched_one_way, row_origin in provisional_rows:
            t_coords = str(brow.get("target_coords") or "")
            if matched_one_way is not None:
                one_way_s = matched_one_way
                t_src = "observed_dispatch_pair"
            elif emp_scale is not None:
                dist = max(1.0, float(_parse_coords_distance(row_origin, t_coords)))
                one_way_s = 10.0 + emp_scale * math.sqrt(dist)
                t_src = "calibrated_distance"
            else:
                one_way_s = estimate_physics_one_way_seconds(row_origin, t_coords)
                t_src = "physics_formula"

            pig = float(brow.get("pig_iron") or 0.0)
            met = float(brow.get("metal") or 0.0)
            kry = float(brow.get("kryptonite") or 0.0)
            spi = float(brow.get("spice") or 0.0)
            sp_eq = float(
                brow.get("loot_spice_eq")
                if brow.get("loot_spice_eq") is not None
                else game_data.spice_equivalent({"pig_iron": pig, "metal": met, "kryptonite": kry, "spice": spi})
            )
            lost_sp = float(brow.get("attacker_lost_spice") or 0.0)
            debris_sp = float(brow.get("debris_spice_eq") or 0.0)
            outcome_str = str(brow.get("outcome") or "").strip().lower()
            won = outcome_str in ("win", "victory")
            is_spacia = bool(brow.get("target_is_spacia"))
            is_inactive = bool(brow.get("target_is_inactive")) or str(brow.get("target_status_color") or "") == "yellow"
            target_class = "spacia" if is_spacia else ("yellow_inactive" if is_inactive else "active_colony")
            records.append(
                {
                    "event_ts": impact_ts,
                    "origin_coords": row_origin,
                    "target_coords": t_coords,
                    "target_class": target_class,
                    "won": won,
                    "outcome": outcome_str or ("win" if won else "loss"),
                    "pig_iron": pig,
                    "metal": met,
                    "kryptonite": kry,
                    "spice": spi,
                    "spice_eq": sp_eq,
                    "attacker_lost_spice": lost_sp,
                    "net_spice_eq": sp_eq - lost_sp,
                    "debris_spice_eq": debris_sp,
                    "one_way_sec": float(one_way_s),
                    "round_trip_sec": float(2.0 * one_way_s),
                    "time_source": t_src,
                }
            )
    else:
        for rrow in raid_rows:
            t_coords = str(rrow.get("target_coords") or "")
            row_origin = str(rrow.get("origin_coords") or origin_coords)
            loot_d = _safe_json_dict(rrow.get("loot") if "loot" in rrow else rrow.get("loot_json"))
            pig = float(loot_d.get("pig_iron") or 0.0)
            met = float(loot_d.get("metal") or 0.0)
            kry = float(loot_d.get("kryptonite") or 0.0)
            spi = float(loot_d.get("spice") or 0.0)
            sp_eq = float(
                game_data.spice_equivalent(
                    {"pig_iron": pig, "metal": met, "kryptonite": kry, "spice": spi}
                )
            )
            outcome_txt = redact_sensitive(str(rrow.get("outcome") or "")).lower()
            won = bool(rrow.get("won")) if "won" in rrow else ("defeat" not in outcome_txt and "loss" not in outcome_txt)
            one_way_s = estimate_physics_one_way_seconds(row_origin, t_coords)
            records.append(
                {
                    "event_ts": float(rrow.get("attacked_ts") or 0.0),
                    "origin_coords": row_origin,
                    "target_coords": t_coords,
                    "target_class": "exempt_farm" if bool(rrow.get("is_exempt")) else "active_colony",
                    "won": won,
                    "outcome": "win" if won else "loss",
                    "pig_iron": pig,
                    "metal": met,
                    "kryptonite": kry,
                    "spice": spi,
                    "spice_eq": sp_eq,
                    "attacker_lost_spice": 0.0,
                    "net_spice_eq": sp_eq,
                    "debris_spice_eq": 0.0,
                    "one_way_sec": float(one_way_s),
                    "round_trip_sec": float(2.0 * one_way_s),
                    "time_source": "physics_formula",
                }
            )

    records.sort(key=lambda r: float(r["event_ts"]))
    if not dispatch_timestamps:
        dispatch_timestamps = [float(r["event_ts"]) for r in records if float(r["event_ts"]) > 0.0]
    return {
        "records": records,
        "dispatch_timestamps": dispatch_timestamps,
        "origin_coords": origin_coords,
        "colony_temperature": colony_temperature,
        "colony_buildings": colony_buildings,
        "colony_researches": colony_researches,
        "universe_speedup_factor": universe_speedup_factor,
        "fleet_base_level": fleet_base_level,
        "fleet_slot_capacity": 1 + fleet_base_level,
        "engine_tech_levels": engine_tech_levels,
        "recycle_rows": recycle_rows,
        "active_bash_blocks": active_bash_blocks,
        "source_label": source_label,
    }


telemetry_bundle = load_attack_telemetry_readonly(LIVE_DB_PATH, ATTACK_HISTORY_PATH)
attack_records: list[dict[str, Any]] = telemetry_bundle["records"]
attack_dispatch_timestamps: list[float] = telemetry_bundle["dispatch_timestamps"]
origin_planet_coords: str = telemetry_bundle["origin_coords"]
fleet_slot_capacity: int = int(telemetry_bundle["fleet_slot_capacity"])
recycle_records: list[dict[str, Any]] = telemetry_bundle["recycle_rows"]
telemetry_source: str = telemetry_bundle["source_label"]

print(
    render_table(
        ["MER Telemetry Property", "Value"],
        [
            ["Telemetry Source", telemetry_source],
            ["Primary Origin Colony", f"{origin_planet_coords} ({telemetry_bundle['colony_temperature']} °C, {telemetry_bundle['universe_speedup_factor']:.1f}x speed)"],
            ["Fleet Base Level / Flight Slots", f"b10 L{telemetry_bundle['fleet_base_level']} -> {fleet_slot_capacity} concurrent slots"],
            ["Engine Tech Levels (r4 / r5 / r6)", f"{telemetry_bundle['engine_tech_levels']['r4']} / {telemetry_bundle['engine_tech_levels']['r5']} / {telemetry_bundle['engine_tech_levels']['r6']}"],
            ["Joined Combat Attack Records", len(attack_records)],
            ["Distinct Targets / Active Bash Blocks", f"{len({r['target_coords'] for r in attack_records})} targets / {telemetry_bundle['active_bash_blocks']} blocked"],
            ["Archived Recycle Harvest Reports", len(recycle_records)],
        ],
    )
)

## 3. Mean & Standard Deviation of Resources Achieved in Attacks (Gross, Net & Rubble)

Using the `BATTLE_REPORTS` schema from [docs/database-mer.md](../docs/database-mer.md), this section computes the **sample mean ($\mu_r$)** and **standard deviation ($s_r$)** across:

- **Gross Plunder**: `pig_iron`, `metal`, `kryptonite`, `spice`, and `spice_eq` (`6 : 2.5 : 3 : 1` Merchant parity).
- **Net Combat Yield (`net_spice_eq`)**: $\text{loot\_spice\_eq} - \text{attacker\_lost\_spice}$, accounting for fleet losses suffered in unsuccessful or costly engagements.
- **Orbital Rubble (`debris_spice_eq` + `RECYCLE_REPORTS`)**: Combat-spawned debris fields and verified `Recycler` (`s11`) harvests (`harvested_spice_eq`).

In [ ]:
METRIC_KEYS: tuple[str, ...] = (*RESOURCE_KEYS, "spice_eq", "net_spice_eq", "debris_spice_eq")
winning_attack_records: list[dict[str, Any]] = [r for r in attack_records if bool(r.get("won"))]

attack_resource_mean: dict[str, float] = {}
attack_resource_std: dict[str, float] = {}
attack_resource_pop_std: dict[str, float] = {}
winning_attack_resource_mean: dict[str, float] = {}
winning_attack_resource_std: dict[str, float] = {}

resource_stat_rows: list[list[Any]] = []
for key in METRIC_KEYS:
    all_vals = [float(r.get(key) or 0.0) for r in attack_records]
    win_vals = [float(r.get(key) or 0.0) for r in winning_attack_records]

    mu_all, sd_all, psd_all = compute_mean_and_std(all_vals)
    mu_win, sd_win, _ = compute_mean_and_std(win_vals)

    attack_resource_mean[key] = mu_all
    attack_resource_std[key] = sd_all
    attack_resource_pop_std[key] = psd_all
    winning_attack_resource_mean[key] = mu_win
    winning_attack_resource_std[key] = sd_win

    resource_stat_rows.append(
        [
            key,
            f"{mu_all:,.2f}",
            f"{sd_all:,.2f}",
            f"{psd_all:,.2f}",
            f"{mu_win:,.2f}",
            f"{sd_win:,.2f}",
        ]
    )

recycle_spice_eq_mean, recycle_spice_eq_std, _ = compute_mean_and_std(
    [float(r.get("harvested_spice_eq") or 0.0) for r in recycle_records]
)

print("3. Attack Resource Loot & Net Combat Yield (All Attacks vs. Winning Attacks)")
print(
    render_table(
        [
            "Metric / Resource",
            "Mean (All)",
            "Sample Std (All)",
            "Pop Std (All)",
            "Mean (Wins Only)",
            "Sample Std (Wins Only)",
        ],
        resource_stat_rows,
    )
)
if recycle_records:
    print(
        f"Supplementary RECYCLE_REPORTS ({len(recycle_records)} harvests): "
        f"mean={recycle_spice_eq_mean:,.2f} SpiceEq, sample_std={recycle_spice_eq_std:,.2f} SpiceEq"
    )

## 4. Mean & Standard Deviation of Attack Time

To accurately simulate fleet-slot capacity (`1 + Fleet base b10` concurrent flights) and raid throughput per hour, we measure two complementary time distributions:

1. **Fleet Mission Flight Time (`one_way_sec` & `round_trip_sec`)**:
   - `one_way_sec` ($\mu_{t,\text{one-way}}, s_{t,\text{one-way}}$): Elapsed flight time from dispatch at origin colony to combat impact at target coordinates (measured directly from paired `raid_history -> battle_reports` timestamps when available, or SpaceInvasion Classic flight physics otherwise).
   - `round_trip_sec` ($\mu_{t,\text{round-trip}}, s_{t,\text{round-trip}}$): Full round-trip sortie duration ($2 \times t_{\text{one-way}}$), which is the exact duration a fleet slot remains occupied before plunder is unloaded at home.
2. **Inter-Attack Dispatch Cadence (`inter_attack_sec`)**:
   - Mean and standard deviation of real wall-clock time ($\Delta t_k = t_{k} - t_{k-1}$) between consecutive attack dispatches in historical telemetry.

In [ ]:
one_way_times_sec: list[float] = [float(r["one_way_sec"]) for r in attack_records]
round_trip_times_sec: list[float] = [float(r["round_trip_sec"]) for r in attack_records]
inter_attack_intervals_sec: list[float] = [
    attack_dispatch_timestamps[idx] - attack_dispatch_timestamps[idx - 1]
    for idx in range(1, len(attack_dispatch_timestamps))
    if attack_dispatch_timestamps[idx] > attack_dispatch_timestamps[idx - 1]
]

attack_one_way_time_mean_sec, attack_one_way_time_std_sec, attack_one_way_time_pop_std_sec = (
    compute_mean_and_std(one_way_times_sec)
)
attack_time_mean_sec, attack_time_std_sec, attack_time_pop_std_sec = (
    compute_mean_and_std(round_trip_times_sec)
)
inter_attack_time_mean_sec, inter_attack_time_std_sec, inter_attack_time_pop_std_sec = (
    compute_mean_and_std(inter_attack_intervals_sec)
)

time_stat_rows: list[list[Any]] = [
    [
        "One-Way Flight Time (Dispatch -> Impact)",
        len(one_way_times_sec),
        f"{attack_one_way_time_mean_sec:,.1f}s ({fmt_duration(attack_one_way_time_mean_sec)})",
        f"{attack_one_way_time_std_sec:,.1f}s ({fmt_duration(attack_one_way_time_std_sec)})",
        f"{attack_one_way_time_pop_std_sec:,.1f}s",
    ],
    [
        "Round-Trip Attack Time (Slot Occupation)",
        len(round_trip_times_sec),
        f"{attack_time_mean_sec:,.1f}s ({fmt_duration(attack_time_mean_sec)})",
        f"{attack_time_std_sec:,.1f}s ({fmt_duration(attack_time_std_sec)})",
        f"{attack_time_pop_std_sec:,.1f}s",
    ],
    [
        "Inter-Attack Dispatch Interval",
        len(inter_attack_intervals_sec),
        f"{inter_attack_time_mean_sec:,.1f}s ({fmt_duration(inter_attack_time_mean_sec)})",
        f"{inter_attack_time_std_sec:,.1f}s ({fmt_duration(inter_attack_time_std_sec)})",
        f"{inter_attack_time_pop_std_sec:,.1f}s",
    ],
]

print("4. Attack Time & Cadence Statistics")
print(
    render_table(
        ["Time Metric", "Samples", "Mean", "Sample Std", "Pop Std"],
        time_stat_rows,
    )
)

## 5. Attack Success Rate

This section evaluates the empirical **Attack Success Rate** across all resolved combat engagements:

$$\hat{p}_{\text{win}} = \frac{W}{N}, \qquad \text{SE}(\hat{p}_{\text{win}}) = \sqrt{\frac{\hat{p}_{\text{win}}(1 - \hat{p}_{\text{win}})}{N}}, \qquad \tilde{p}_{\text{win}} = \frac{W + 1}{N + 2}$$

- **`attack_success_rate` ($\hat{p}_{\text{win}}$)**: Observed fraction of attacks ending in combat victory (`won == True`).
- **`attack_success_rate_std_err`**: Binomial standard error quantifying sample uncertainty around $\hat{p}_{\text{win}}$.
- **`attack_success_rate_smoothed` ($\tilde{p}_{\text{win}}$)**: Laplace-smoothed (`Beta(1, 1)`) estimate preventing degenerate `0%` or `100%` priors when `N` is small.
- **`ATTACK_SIM_PARAMS`**: Bundles the resource means/stds, time means/stds, success rates, and expected `SpiceEq` yield per fleet-slot hour into a single dictionary ready for downstream full-planner simulations.

In [ ]:
attack_attempts: int = len(attack_records)
attack_wins: int = sum(1 for r in attack_records if bool(r.get("won")))
attack_losses: int = attack_attempts - attack_wins

attack_success_rate: float = (
    float(attack_wins) / float(attack_attempts) if attack_attempts > 0 else 0.0
)
attack_success_rate_std_err: float = (
    math.sqrt(attack_success_rate * (1.0 - attack_success_rate) / float(attack_attempts))
    if attack_attempts > 0
    else 0.0
)
attack_success_rate_smoothed: float = float(attack_wins + 1) / float(attack_attempts + 2)

per_target_success_rate: dict[str, dict[str, Any]] = {}
per_class_success_rate: dict[str, dict[str, Any]] = {}
for rec in attack_records:
    coords = str(rec.get("target_coords") or "unknown")
    t_cls = str(rec.get("target_class") or "unknown")
    for group_map, group_key in ((per_target_success_rate, coords), (per_class_success_rate, t_cls)):
        bucket = group_map.setdefault(
            group_key,
            {"attempts": 0, "wins": 0, "losses": 0, "loot_spice_eq": [], "net_spice_eq": []},
        )
        bucket["attempts"] += 1
        if bool(rec.get("won")):
            bucket["wins"] += 1
        else:
            bucket["losses"] += 1
        bucket["loot_spice_eq"].append(float(rec.get("spice_eq") or 0.0))
        bucket["net_spice_eq"].append(float(rec.get("net_spice_eq") or 0.0))

target_sr_rows: list[list[Any]] = []
for coords in sorted(per_target_success_rate.keys()):
    b = per_target_success_rate[coords]
    att = int(b["attempts"])
    wins = int(b["wins"])
    sr = float(wins) / float(att) if att > 0 else 0.0
    mu_sp, sd_sp, _ = compute_mean_and_std(list(b["loot_spice_eq"]))
    mu_net, _, _ = compute_mean_and_std(list(b["net_spice_eq"]))
    b["success_rate"] = sr
    b["mean_spice_eq"] = mu_sp
    b["std_spice_eq"] = sd_sp
    b["mean_net_spice_eq"] = mu_net
    target_sr_rows.append(
        [
            coords,
            f"{wins} / {att}",
            f"{sr:.2%}",
            f"{mu_sp:,.2f}",
            f"{sd_sp:,.2f}",
            f"{mu_net:+,.2f}",
        ]
    )

class_sr_rows: list[list[Any]] = []
for t_cls in sorted(per_class_success_rate.keys()):
    cb = per_class_success_rate[t_cls]
    att = int(cb["attempts"])
    wins = int(cb["wins"])
    sr = float(wins) / float(att) if att > 0 else 0.0
    mu_sp, _, _ = compute_mean_and_std(list(cb["loot_spice_eq"]))
    mu_net, _, _ = compute_mean_and_std(list(cb["net_spice_eq"]))
    cb["success_rate"] = sr
    cb["mean_spice_eq"] = mu_sp
    cb["mean_net_spice_eq"] = mu_net
    class_sr_rows.append([t_cls, f"{wins} / {att}", f"{sr:.2%}", f"{mu_sp:,.2f}", f"{mu_net:+,.2f}"])

expected_raids_per_slot_hr: float = (
    3600.0 / attack_time_mean_sec if attack_time_mean_sec > 0.0 else 0.0
)
expected_spice_eq_per_slot_hr: float = (
    expected_raids_per_slot_hr * attack_resource_mean["spice_eq"]
)
expected_net_spice_eq_per_slot_hr: float = (
    expected_raids_per_slot_hr * attack_resource_mean["net_spice_eq"]
)
max_empire_gross_spice_eq_hr: float = expected_spice_eq_per_slot_hr * float(fleet_slot_capacity)

ATTACK_SIM_PARAMS: dict[str, Any] = {
    "attempts": attack_attempts,
    "wins": attack_wins,
    "losses": attack_losses,
    "success_rate": attack_success_rate,
    "success_rate_std_err": attack_success_rate_std_err,
    "success_rate_smoothed": attack_success_rate_smoothed,
    "fleet_slot_capacity": fleet_slot_capacity,
    "resource_mean": dict(attack_resource_mean),
    "resource_std": dict(attack_resource_std),
    "winning_resource_mean": dict(winning_attack_resource_mean),
    "winning_resource_std": dict(winning_attack_resource_std),
    "recycle_spice_eq_mean": recycle_spice_eq_mean,
    "recycle_spice_eq_std": recycle_spice_eq_std,
    "one_way_time_mean_sec": attack_one_way_time_mean_sec,
    "one_way_time_std_sec": attack_one_way_time_std_sec,
    "round_trip_time_mean_sec": attack_time_mean_sec,
    "round_trip_time_std_sec": attack_time_std_sec,
    "inter_attack_time_mean_sec": inter_attack_time_mean_sec,
    "inter_attack_time_std_sec": inter_attack_time_std_sec,
    "expected_raids_per_slot_hr": expected_raids_per_slot_hr,
    "expected_spice_eq_per_slot_hr": expected_spice_eq_per_slot_hr,
    "expected_net_spice_eq_per_slot_hr": expected_net_spice_eq_per_slot_hr,
    "max_empire_gross_spice_eq_hr": max_empire_gross_spice_eq_hr,
}

print("5. Attack Success Rate & Expected Fleet-Slot Yield")
print(
    render_table(
        ["Metric", "Value"],
        [
            ["Total Attacks (Wins / Losses)", f"{attack_attempts} ({attack_wins} wins / {attack_losses} losses)"],
            ["Empirical Attack Success Rate", f"{attack_success_rate:.2%} (+/- {attack_success_rate_std_err:.2%} SE)"],
            ["Laplace-Smoothed Success Rate", f"{attack_success_rate_smoothed:.2%}"],
            ["Max Sorties per Fleet Slot / Hour", f"{expected_raids_per_slot_hr:.2f} raids/hr (mean round-trip {fmt_duration(attack_time_mean_sec)})"],
            ["Gross / Net Loot per Slot-Hour", f"{expected_spice_eq_per_slot_hr:,.1f} gross / {expected_net_spice_eq_per_slot_hr:+,.1f} net SpiceEq/hr"],
            [f"Full Capacity Ceiling ({fleet_slot_capacity} Slots)", f"{max_empire_gross_spice_eq_hr:,.1f} gross SpiceEq/hr"],
        ],
    )
)
if class_sr_rows:
    print("\nSuccess Rate by SPY_TARGETS Class (MER Join)")
    print(
        render_table(
            ["Target Class", "Wins / Attempts", "Success Rate", "Mean Gross SpEq", "Mean Net SpEq"],
            class_sr_rows,
        )
    )
if target_sr_rows:
    print("\nPer-Target Success Rate & Net Combat Yield")
    print(
        render_table(
            ["Target Coords", "Wins / Attempts", "Success Rate", "Mean Gross SpEq", "Sample Std SpEq", "Mean Net SpEq"],
            target_sr_rows,
        )
    )

## 6. Game Parameters Catalog (`pandas` DataFrames) — Reference Context & Merchant Parity

This section builds the complete **`pandas` DataFrame parameter database** for SpaceInvasion Classic from `spaceinvasion.game_data`, normalizing all buildings (`b1`–`b21`), technologies (`r1`–`r15`), spaceships (`s1`–`s16`), and planetary defenses (`d1`–`d11`) for downstream strategy simulation.

First, we establish `GAME_PARAM_CONTEXT` (capturing the active universe `speedup_factor`, colony `temperature`, local construction/research/shipyard accelerators `b7` / `b8` / `b9` / `b17`, and global tech levels `r1`–`r15` from SQLite telemetry) alongside **`merchant_rates_df`**, which encodes fixed Merchant exchange parity (`6 Pig iron : 2.5 Metal : 3 Kryptonite : 1 Spice`) and the global ranking divisor (`1 point = 1,000 invested resources`).

In [ ]:
MAX_BUILDING_TABLE_LEVEL: int = 50
MAX_RESEARCH_TABLE_LEVEL: int = 30

_col_bld = telemetry_bundle.get("colony_buildings") or {}
_col_res = telemetry_bundle.get("colony_researches") or {}

GAME_PARAM_CONTEXT: dict[str, Any] = {
    "wiki_synced_at_utc": game_data.WIKI_SYNCED_AT_UTC,
    "speedup_factor": float(telemetry_bundle.get("universe_speedup_factor") or 5.0),
    "planet_temperature_c": int(telemetry_bundle.get("colony_temperature", 31)),
    "b7_dev_center": max(0, int(_col_bld.get("b7") or 0)),
    "b8_research_lab": max(0, int(_col_bld.get("b8") or 0)),
    "b9_weapons_factory": max(0, int(_col_bld.get("b9") or 0)),
    "b17_microsystem": max(0, int(_col_bld.get("b17") or 0)),
    "r1_mining_tech": max(0, int(_col_res.get("r1") or 0)),
    "r2_energy_tech": max(0, int(_col_res.get("r2") or 0)),
    "r4_rocket_engine": max(0, int(_col_res.get("r4") or 0)),
    "r5_diffusion_engine": max(0, int(_col_res.get("r5") or 0)),
    "r6_warp_engine": max(0, int(_col_res.get("r6") or 0)),
    "r8_weapons_tech": max(0, int(_col_res.get("r8") or 0)),
    "r9_shield_tech": max(0, int(_col_res.get("r9") or 0)),
    "r10_defense_systems": max(0, int(_col_res.get("r10") or 0)),
    "points_per_resource_divisor": float(POINTS_PER_RESOURCE_DIVISOR),
}

BUILDING_TIME_DIVISOR: float = max(
    1.0,
    float(GAME_PARAM_CONTEXT["b7_dev_center"] + 1)
    * float(2 ** min(GAME_PARAM_CONTEXT["b17_microsystem"], 40))
    * float(GAME_PARAM_CONTEXT["speedup_factor"]),
)
RESEARCH_TIME_DIVISOR: float = max(
    1.0,
    float(GAME_PARAM_CONTEXT["b8_research_lab"] + 1)
    * float(GAME_PARAM_CONTEXT["speedup_factor"]),
)
SHIPYARD_TIME_DIVISOR: float = max(
    1.0,
    float(max(1, GAME_PARAM_CONTEXT["b9_weapons_factory"]))
    * float(2 ** min(GAME_PARAM_CONTEXT["b17_microsystem"], 40))
    * float(GAME_PARAM_CONTEXT["speedup_factor"]),
)

_RESOURCE_NOTES: dict[str, str] = {
    "pig_iron": "Direct planetary mine extraction (+2%/lvl r1); feedstock for Blast furnace (2:1)",
    "metal": "Smelted 2:1 from Pig iron inside Blast furnace (b2); NOT boosted by r1",
    "kryptonite": "Direct planetary mine extraction (+2%/lvl r1); heavy tech & warship input",
    "spice": "Temperature-interpolated output (higher on colder colonies) +2%/lvl r1; fleet & scan fuel",
}

merchant_rates_df = pd.DataFrame(
    [
        {
            "resource": rk,
            "units_per_1_spice": float(game_data.MERCHANT_RATES_IN_SPICE[rk]),
            "spice_eq_per_unit": round(1.0 / float(game_data.MERCHANT_RATES_IN_SPICE[rk]), 6),
            "points_per_unit": 1.0 / float(POINTS_PER_RESOURCE_DIVISOR),
            "mine_building_key": f"b{idx + 1}",
            "boosted_by_mining_tech_r1": rk != "metal",
            "notes": _RESOURCE_NOTES[rk],
        }
        for idx, rk in enumerate(RESOURCE_KEYS)
    ]
)

print(
    f"Game Parameter Context: speed={GAME_PARAM_CONTEXT['speedup_factor']:.1f}x, "
    f"temp={GAME_PARAM_CONTEXT['planet_temperature_c']}°C | "
    f"Divisors -> Build={BUILDING_TIME_DIVISOR:.1f}x (b7 L{GAME_PARAM_CONTEXT['b7_dev_center']}, b17 L{GAME_PARAM_CONTEXT['b17_microsystem']}), "
    f"Research={RESEARCH_TIME_DIVISOR:.1f}x (b8 L{GAME_PARAM_CONTEXT['b8_research_lab']}), "
    f"Shipyard={SHIPYARD_TIME_DIVISOR:.1f}x (b9 L{GAME_PARAM_CONTEXT['b9_weapons_factory']})"
)
print("\n[merchant_rates_df]")
print(merchant_rates_df.to_string(index=False))

### 6.1 Buildings Parameters (`b1`–`b21`) — Costs, Build Times, Production, Energy & Storage (`buildings_catalog_df`, `buildings_levels_df`)

This component materializes two complementary `pandas.DataFrame` tables covering all **21 SpaceInvasion Classic buildings**:

1. **`buildings_catalog_df` (`21` rows, `b1`–`b21`)**: Master metadata per building including `category`, `asteroid_ok`, `max_table_level`, `cost_scale_factor`, `L1` resource bundle (`pig_iron`, `metal`, `kryptonite`, `spice`, `l1_cost_spice_eq`), `L1` base and colony-effective build times, tech-tree `prerequisites`, and `primary_effect`.
2. **`buildings_levels_df` (`L1..L50` per building)**: Exact level-by-level progression tracking:
   - **Costs & Points**: `pig_iron`, `metal`, `kryptonite`, `spice`, `cost_spice_eq`, and `points_gain` (`cost_sum / 1,000`).
   - **Construction Duration**: `base_time_s` (`1x` Wiki baseline) and `effective_build_time_s = base_time_s / ((b7 + 1) * 2^b17 * speedup_factor)`.
   - **Hourly Production & Feedstock**: `hourly_prod_1x`, `hourly_prod_effective` (`+2%/lvl` `r1` on `b1`/`b3`/`b4`, temperature-interpolated for `b4` *Spice mine*; unboosted for `b2` *Blast furnace*), `pig_iron_consumed_hr_effective` (`2:1` smelting feedstock on `b2`), `spice_consumed_hr` (`b6` *Thermosolar* fuel), marginal `delta_net_spice_eq_hr`, and incremental `marginal_roi_hours`.
   - **Energy, Storage & Accelerators**: `energy_at_level_base`, `energy_at_level_effective` (`+1%/lvl` `r2` on `b5`/`b6`), `energy_delta_from_prev`, warehouse `storage_capacity` (`100,000 * 2^L` on `b11`–`b14`), and `time_reduction_multiplier` (`1/(L+1)` on `b7`, `1/L` on `b9`, `0.5^L` on `b17`).

In [ ]:
_BUILDING_META: dict[int, tuple[str, float, str]] = {
    1: ("resource_mine", 1.51, "Mines Pig iron (+2%/lvl r1); consumes energy"),
    2: ("smelter", 1.51, "Smelts Pig iron -> Metal at 2:1 loss (NOT boosted by r1); consumes energy"),
    3: ("resource_mine", 1.51, "Mines Kryptonite (+2%/lvl r1); consumes energy"),
    4: ("resource_mine", 1.51, "Mines Spice on planet-temperature curve (+2%/lvl r1); consumes energy"),
    5: ("power_plant", 1.51, "Fuel-free planetary power (+1%/lvl r2); 1 field/level"),
    6: ("power_plant", 1.51, "Spice-burning planetary power (+1%/lvl r2); forbidden on normal planets when forbid_planetary_solar=True"),
    7: ("accelerator", 2.00, "Cuts planetary building construction time by 1 / (L + 1); gates b8, b9, b17, b19"),
    8: ("research_infra", 2.00, "Unlocks & accelerates global research time by 1 / (L + 1)"),
    9: ("shipyard", 2.00, "Unlocks ships/defenses & cuts unit build time by 1 / max(1, L)"),
    10: ("fleet_infra", 2.00, "Grants 1 + L concurrent fleet flight slots & allied fleet support"),
    11: ("warehouse", 2.00, "Pig iron storage capacity 100,000 * 2^L (prevents 5% overflow penalty)"),
    12: ("warehouse", 2.00, "Metal storage capacity 100,000 * 2^L"),
    13: ("warehouse", 2.00, "Kryptonite storage capacity 100,000 * 2^L"),
    14: ("warehouse", 2.00, "Spice storage capacity 100,000 * 2^L"),
    15: ("strategic_weapon", 2.00, "Stores surplus energy to fire remote particle beams at enemy defenses"),
    16: ("strategic_defense", 2.00, "Absorbs incoming Particle cannon beams (46% / 31% / 23% priority split)"),
    17: ("accelerator", 2.00, "Halves all building & shipyard build times each level (0.5^L)"),
    18: ("fleet_infra", 2.00, "Instant zero-flight-time fleet jump between charged Teleporters"),
    19: ("asteroid_sensor", 2.00, "Asteroid-exclusive sensor scanning enemy fleet flights (5,000 Spice/scan; Escape flights hidden)"),
    20: ("strategic_defense", 1.08, "Cuts defense dig-in time (60m - 1m/L) & adds L40+ particle fog"),
    21: ("endgame_synth", 1.44, "Grants +1 Synth Point per level (up to L30, highest planet only)"),
}

_PRODUCED_RES_BY_BID: dict[int, str] = {
    1: "pig_iron",
    2: "metal",
    3: "kryptonite",
    4: "spice",
}

_speedup_f = float(GAME_PARAM_CONTEXT["speedup_factor"])
_temp_c = int(GAME_PARAM_CONTEXT["planet_temperature_c"])
_r1_lvl = int(GAME_PARAM_CONTEXT["r1_mining_tech"])
_r2_lvl = int(GAME_PARAM_CONTEXT["r2_energy_tech"])
_energy_mult = 1.0 + 0.01 * float(_r2_lvl)

_b_cat_rows: list[dict[str, Any]] = []
_b_lvl_rows: list[dict[str, Any]] = []

for _bid in sorted(game_data.BUILDING_BY_ID.keys()):
    _info = game_data.BUILDING_BY_ID[_bid]
    _bkey = f"b{_bid}"
    _cat, _scale, _effect = _BUILDING_META.get(_bid, ("other", 2.0, ""))
    _max_tbl = int(_info.get("max_table_level") or len(_info.get("levels") or {}))
    _lvls_map = _info.get("levels") or {}
    _mono_times = build_monotonic_base_times(_lvls_map, _max_tbl)
    _l1_cost = game_data.building_cost(_bid, 1)
    _l1_sp = float(game_data.spice_equivalent(_l1_cost))
    _l1_bt = float(_mono_times.get(1, 0.0))

    _b_cat_rows.append(
        {
            "building_id": _bid,
            "key": _bkey,
            "slug": str(_info["slug"]),
            "name": str(_info["name"]),
            "category": _cat,
            "asteroid_ok": bool(_info.get("asteroid_ok", False)),
            "max_table_level": _max_tbl,
            "cost_scale_factor": _scale,
            "l1_pig_iron": int(_l1_cost["pig_iron"]),
            "l1_metal": int(_l1_cost["metal"]),
            "l1_kryptonite": int(_l1_cost["kryptonite"]),
            "l1_spice": int(_l1_cost["spice"]),
            "l1_cost_spice_eq": round(_l1_sp, 2),
            "l1_base_time_s": round(_l1_bt, 2),
            "l1_effective_time_s": round(_l1_bt / BUILDING_TIME_DIVISOR, 2),
            "prerequisites": format_prerequisites(_info.get("requirements") or []),
            "primary_effect": _effect,
        }
    )

    _prev_net_sp_hr = 0.0
    _prev_energy_eff = 0.0
    _sim_cap = min(_max_tbl, MAX_BUILDING_TABLE_LEVEL)
    for _lvl in range(1, _sim_cap + 1):
        _row = _lvls_map.get(_lvl) or {}
        _cost = game_data.building_cost(_bid, _lvl)
        _cost_sp = float(game_data.spice_equivalent(_cost))
        _pts_gain = float(sum(_cost.values())) / float(POINTS_PER_RESOURCE_DIVISOR)
        _bt_s = float(_mono_times.get(_lvl, 0.0))
        _eff_t_s = _bt_s / BUILDING_TIME_DIVISOR

        _prod_res = _PRODUCED_RES_BY_BID.get(_bid, "none")
        if _bid in (1, 2, 3, 4):
            _prod_1x = float(
                game_data.mine_production_per_hour(_bid, _lvl, temperature=_temp_c, mining_tech_level=0)
            )
            _prod_eff = float(
                game_data.mine_production_per_hour(_bid, _lvl, temperature=_temp_c, mining_tech_level=_r1_lvl)
            ) * _speedup_f
        else:
            _prod_1x = 0.0
            _prod_eff = 0.0

        _pig_used_1x = float(game_data.blast_furnace_pig_iron_used(_lvl)) if _bid == 2 else 0.0
        _pig_used_eff = _pig_used_1x * _speedup_f
        _spice_used_hr = float(_row.get("spice_used") or 0.0) if _bid == 6 else 0.0

        _gross_sp_hr = (
            _prod_eff / float(game_data.MERCHANT_RATES_IN_SPICE[_prod_res])
            if _prod_res in game_data.MERCHANT_RATES_IN_SPICE
            else 0.0
        )
        _feedstock_sp_hr = (
            _pig_used_eff / float(game_data.MERCHANT_RATES_IN_SPICE["pig_iron"])
            + _spice_used_hr / float(game_data.MERCHANT_RATES_IN_SPICE["spice"])
        )
        _net_sp_hr = _gross_sp_hr - _feedstock_sp_hr
        _delta_net_sp_hr = _net_sp_hr - _prev_net_sp_hr
        _roi_h = (_cost_sp / _delta_net_sp_hr) if _delta_net_sp_hr > 0.0 else float("inf")
        _prev_net_sp_hr = _net_sp_hr

        _e_base = float(_row.get("energy") or 0.0)
        _e_eff = (_e_base * _energy_mult) if _e_base > 0.0 else _e_base
        _e_delta = _e_eff - _prev_energy_eff
        _prev_energy_eff = _e_eff

        _storage_cap = int(_row.get("storage_capacity") or (100_000 * (2**_lvl))) if _bid in (11, 12, 13, 14) else 0
        if _bid == 7:
            _time_mult = 1.0 / float(_lvl + 1)
        elif _bid == 8:
            _time_mult = 1.0 / float(_lvl + 1)
        elif _bid == 9:
            _time_mult = 1.0 / float(max(1, _lvl))
        elif _bid == 17:
            _time_mult = 0.5 ** float(_lvl)
        else:
            _time_mult = 1.0

        _b_lvl_rows.append(
            {
                "building_id": _bid,
                "key": _bkey,
                "slug": str(_info["slug"]),
                "name": str(_info["name"]),
                "category": _cat,
                "level": _lvl,
                "pig_iron": int(_cost["pig_iron"]),
                "metal": int(_cost["metal"]),
                "kryptonite": int(_cost["kryptonite"]),
                "spice": int(_cost["spice"]),
                "cost_spice_eq": round(_cost_sp, 2),
                "points_gain": round(_pts_gain, 3),
                "base_time_s": round(_bt_s, 2),
                "effective_build_time_s": round(_eff_t_s, 2),
                "effective_build_time_text": fmt_duration(_eff_t_s),
                "produced_resource": _prod_res,
                "hourly_prod_1x": round(_prod_1x, 2),
                "hourly_prod_effective": round(_prod_eff, 2),
                "pig_iron_consumed_hr_effective": round(_pig_used_eff, 2),
                "spice_consumed_hr": round(_spice_used_hr, 2),
                "net_spice_eq_hr_effective": round(_net_sp_hr, 2),
                "delta_net_spice_eq_hr": round(_delta_net_sp_hr, 2),
                "marginal_roi_hours": round(_roi_h, 2) if math.isfinite(_roi_h) else None,
                "energy_at_level_base": round(_e_base, 2),
                "energy_at_level_effective": round(_e_eff, 2),
                "energy_delta_from_prev": round(_e_delta, 2),
                "storage_capacity": _storage_cap,
                "time_reduction_multiplier": round(_time_mult, 6),
            }
        )

buildings_catalog_df = pd.DataFrame(_b_cat_rows)
buildings_levels_df = pd.DataFrame(_b_lvl_rows)

print(
    f"Built buildings_catalog_df ({buildings_catalog_df.shape[0]} rows x {buildings_catalog_df.shape[1]} cols) "
    f"and buildings_levels_df ({buildings_levels_df.shape[0]} rows x {buildings_levels_df.shape[1]} cols)"
)
print("\n[buildings_catalog_df — All 21 Buildings]")
print(
    buildings_catalog_df[
        [
            "key",
            "name",
            "category",
            "asteroid_ok",
            "cost_scale_factor",
            "l1_cost_spice_eq",
            "l1_effective_time_s",
            "prerequisites",
        ]
    ].to_string(index=False)
)
print("\n[buildings_levels_df — Economy Checkpoints (b1..b5 at L1, L10, L20)]")
print(
    buildings_levels_df[
        buildings_levels_df["building_id"].isin([1, 2, 3, 4, 5])
        & buildings_levels_df["level"].isin([1, 10, 20])
    ][
        [
            "key",
            "name",
            "level",
            "cost_spice_eq",
            "effective_build_time_text",
            "hourly_prod_effective",
            "net_spice_eq_hr_effective",
            "marginal_roi_hours",
            "energy_at_level_effective",
        ]
    ].to_string(index=False)
)

### 6.2 Technologies Parameters (`r1`–`r15`) — Costs, Research Times & Quantitative Effects (`technologies_catalog_df`, `technologies_levels_df`)

All SpaceInvasion Classic technologies apply **account-wide** across every planet, asteroid, and fleet upon completion, scaling resource costs and base research times by exact doubling (`2.0^(L - 1)`) and accelerating by the local planet's *Research lab* (`b8`) via `effective_research_time_s = base_time_s / ((b8 + 1) * speedup_factor)`.

This component builds two `pandas.DataFrame` tables:

1. **`technologies_catalog_df` (`15` rows, `r1`–`r15`)**: Encodes `domain`, `max_table_level`, `L1` resource costs & `l1_cost_spice_eq`, `L1` base & effective research durations, `effect_metric`, exact **`per_level_bonus_pct`** (`+2%/lvl` `r1` mines excluding `b2`, `+1%/lvl` `r2` global energy, `+10%/lvl` `r4`/`r5`/`r6` ship speed by engine class, `+10%/lvl` `r8` attack / `r9` shield / `r10` hull), `applies_to`, `prerequisites`, and `strategic_effect`.
2. **`technologies_levels_df` (`L1..L30` per research)**: Level-by-level progression with `pig_iron..spice`, `cost_spice_eq`, `points_gain`, `base_time_s`, `effective_research_time_s`, cumulative `bonus_pct_total`, `multiplier_at_level`, and human-readable `effect_at_level`.

In [ ]:
_TECH_EFFECT_SPECS: dict[str, dict[str, Any]] = {
    "r1": {
        "domain": "economy",
        "effect_metric": "mine_output_bonus",
        "per_level_bonus_pct": 2.0,
        "applies_to": "b1 Pig iron, b3 Kryptonite, b4 Spice across all planets (NOT b2 Blast furnace)",
        "strategic_effect": "+2%/lvl global Pig iron, Kryptonite & Spice mine output",
    },
    "r2": {
        "domain": "economy",
        "effect_metric": "global_energy_bonus",
        "per_level_bonus_pct": 1.0,
        "applies_to": "b5 Fusion plant, b6 Thermosolar plant, s9 Solar cells across all planets",
        "strategic_effect": "+1%/lvl global energy output; gates b9, b6, r1, shields & turrets",
    },
    "r3": {
        "domain": "propulsion",
        "effect_metric": "engine_tier_unlock",
        "per_level_bonus_pct": 0.0,
        "applies_to": "Unlocks Rocket (L2), Diffusion (L6), and Warp (L8) drive trees",
        "strategic_effect": "Core propulsion gate (L2 -> r4, L6 -> r5, L8 -> r6)",
    },
    "r4": {
        "domain": "propulsion",
        "effect_metric": "rocket_ship_speed_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "s1 Small transport, s2 Large transport, s4 Small fighter, s5 Warship, s8 Colony ship, s10 Spy probe",
        "strategic_effect": "+10%/lvl flight speed on Rocket-engine ships",
    },
    "r5": {
        "domain": "propulsion",
        "effect_metric": "diffusion_ship_speed_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "s3 Transmitter, s6 Starcruiser, s7 Phoenix, s11 Recycler",
        "strategic_effect": "+10%/lvl flight speed on Diffusion-engine fast-raid & recycler fleets",
    },
    "r6": {
        "domain": "propulsion",
        "effect_metric": "warp_ship_speed_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "s13 Battleship, s14 Stealth bomber, s15 Destroyer, s16 Imperial starbase",
        "strategic_effect": "+10%/lvl flight speed on heavy Warp-engine capital wings",
    },
    "r7": {
        "domain": "intelligence",
        "effect_metric": "espionage_and_stealth",
        "per_level_bonus_pct": 0.0,
        "applies_to": "s10 Spy probe intel depth, counter-espionage, s14 Stealth bomber late detection",
        "strategic_effect": "Expands spy report tiers (fleets/defense/buildings/research) & Stealth concealment",
    },
    "r8": {
        "domain": "combat",
        "effect_metric": "unit_attack_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "All ships (s1..s16) and planetary defenses (d1..d11)",
        "strategic_effect": "+10%/lvl weapon attack power on all combat units",
    },
    "r9": {
        "domain": "combat",
        "effect_metric": "unit_shield_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "All ships (s1..s16) and planetary defenses (d1..d11)",
        "strategic_effect": "+10%/lvl shield absorption capacity; gates Shield domes & r12",
    },
    "r10": {
        "domain": "combat",
        "effect_metric": "unit_hull_bonus",
        "per_level_bonus_pct": 10.0,
        "applies_to": "All ships (s1..s16) and planetary defenses (d1..d11)",
        "strategic_effect": "+10%/lvl structural hull/armor points; gates s7 Phoenix (L5)",
    },
    "r11": {
        "domain": "defense_unlock",
        "effect_metric": "turret_tier_unlock",
        "per_level_bonus_pct": 0.0,
        "applies_to": "Ion, Heavy graviton, Positron, Interval, Laser, and Solar cannons",
        "strategic_effect": "Unlocks mid-to-heavy planetary turret tiers (L4..L10)",
    },
    "r12": {
        "domain": "strategic_weapon",
        "effect_metric": "particle_tech_unlock",
        "per_level_bonus_pct": 0.0,
        "applies_to": "b16 Particle shield (L2), d6 Interval cannon (L4), b15 Particle cannon (L5)",
        "strategic_effect": "Gates planetary Particle shield, Interval cannon & Particle cannon",
    },
    "r13": {
        "domain": "defense_unlock",
        "effect_metric": "graviton_unlock",
        "per_level_bonus_pct": 0.0,
        "applies_to": "Light/Heavy graviton cannons & r15 Black hole research (L10)",
        "strategic_effect": "Unlocks Graviton turrets & gates Black hole research",
    },
    "r14": {
        "domain": "logistics",
        "effect_metric": "teleporter_cooldown_and_unlock",
        "per_level_bonus_pct": 0.0,
        "applies_to": "b18 Teleporter (unlocked at L6) cooldown & energy efficiency",
        "strategic_effect": "Unlocks b18 Teleporter (L6) & reduces jump cooldown/energy",
    },
    "r15": {
        "domain": "endgame_warfare",
        "effect_metric": "scanner_jam_duration_hours",
        "per_level_bonus_pct": 0.0,
        "applies_to": "s16 Imperial starbase unlock (L2) & system scanner-blindness (4h + 1h * L)",
        "strategic_effect": "Unlocks s16 Imperial starbase (L2) & extends scanner jam to (4 + L) hours",
    },
}

_r_by_num: list[tuple[int, str, dict[str, Any]]] = []
for _slug, _rinfo in game_data.RESEARCHES.items():
    _rkey = _rinfo.get("id")
    if isinstance(_rkey, str) and _rkey.startswith("r") and _rkey[1:].isdigit():
        _r_by_num.append((int(_rkey[1:]), _rkey, _rinfo))
_r_by_num.sort(key=lambda item: item[0])

_t_cat_rows: list[dict[str, Any]] = []
_t_lvl_rows: list[dict[str, Any]] = []

for _rnum, _rkey, _rinfo in _r_by_num:
    _spec = _TECH_EFFECT_SPECS.get(_rkey, {})
    _lvls_map = _rinfo.get("levels") or {}
    _max_tbl = int(_rinfo.get("max_table_level") or len(_lvls_map))
    _mono_times = build_monotonic_base_times(_lvls_map, _max_tbl)
    _l1_cost = dict((_lvls_map.get(1) or {}).get("cost") or _rinfo.get("base_cost") or {})
    for _rk in RESOURCE_KEYS:
        _l1_cost.setdefault(_rk, 0)
    _l1_sp = float(game_data.spice_equivalent(_l1_cost))
    _l1_bt = float(_mono_times.get(1, 0.0))
    _bonus_pct = float(_spec.get("per_level_bonus_pct") or 0.0)
    _cur_lvl = max(0, int((telemetry_bundle.get("colony_researches") or {}).get(_rkey) or 0))

    _t_cat_rows.append(
        {
            "research_id": _rnum,
            "key": _rkey,
            "slug": str(_rinfo["slug"]),
            "name": str(_rinfo["name"]),
            "domain": str(_spec.get("domain", "general")),
            "current_level": _cur_lvl,
            "max_table_level": _max_tbl,
            "cost_scale_factor": 2.0,
            "l1_pig_iron": int(_l1_cost["pig_iron"]),
            "l1_metal": int(_l1_cost["metal"]),
            "l1_kryptonite": int(_l1_cost["kryptonite"]),
            "l1_spice": int(_l1_cost["spice"]),
            "l1_cost_spice_eq": round(_l1_sp, 2),
            "l1_base_time_s": round(_l1_bt, 2),
            "l1_effective_time_s": round(_l1_bt / RESEARCH_TIME_DIVISOR, 2),
            "effect_metric": str(_spec.get("effect_metric", "unlock")),
            "per_level_bonus_pct": _bonus_pct,
            "applies_to": str(_spec.get("applies_to", "")),
            "prerequisites": format_prerequisites(_rinfo.get("requirements") or []),
            "strategic_effect": str(_spec.get("strategic_effect", "")),
        }
    )

    _sim_cap = min(_max_tbl, MAX_RESEARCH_TABLE_LEVEL)
    for _lvl in range(1, _sim_cap + 1):
        _row = _lvls_map.get(_lvl) or {}
        _cost = dict(_row.get("cost") or {})
        for _rk in RESOURCE_KEYS:
            _cost.setdefault(_rk, 0)
        _cost_sp = float(game_data.spice_equivalent(_cost))
        _pts_gain = float(sum(int(_cost[k]) for k in RESOURCE_KEYS)) / float(POINTS_PER_RESOURCE_DIVISOR)
        _bt_s = float(_mono_times.get(_lvl, 0.0))
        _eff_t_s = _bt_s / RESEARCH_TIME_DIVISOR
        _tot_pct = _bonus_pct * float(_lvl)
        _mult = 1.0 + (_tot_pct / 100.0)
        if _bonus_pct > 0.0:
            _eff_txt = f"+{_tot_pct:.0f}% ({_mult:.2f}x)"
        elif _rkey == "r15":
            _eff_txt = f"Scanner jam {4 + _lvl}h" + (" + Starbase unlocked" if _lvl >= 2 else "")
        else:
            _eff_txt = f"Tier L{_lvl} unlock / capability"

        _t_lvl_rows.append(
            {
                "research_id": _rnum,
                "key": _rkey,
                "slug": str(_rinfo["slug"]),
                "name": str(_rinfo["name"]),
                "domain": str(_spec.get("domain", "general")),
                "level": _lvl,
                "pig_iron": int(_cost["pig_iron"]),
                "metal": int(_cost["metal"]),
                "kryptonite": int(_cost["kryptonite"]),
                "spice": int(_cost["spice"]),
                "cost_spice_eq": round(_cost_sp, 2),
                "points_gain": round(_pts_gain, 3),
                "base_time_s": round(_bt_s, 2),
                "effective_research_time_s": round(_eff_t_s, 2),
                "effective_research_time_text": fmt_duration(_eff_t_s),
                "bonus_pct_total": round(_tot_pct, 2),
                "multiplier_at_level": round(_mult, 4),
                "effect_at_level": _eff_txt,
            }
        )

technologies_catalog_df = pd.DataFrame(_t_cat_rows)
technologies_levels_df = pd.DataFrame(_t_lvl_rows)

print(
    f"Built technologies_catalog_df ({technologies_catalog_df.shape[0]} rows x {technologies_catalog_df.shape[1]} cols) "
    f"and technologies_levels_df ({technologies_levels_df.shape[0]} rows x {technologies_levels_df.shape[1]} cols)"
)
print("\n[technologies_catalog_df — All 15 Researches & Effects]")
print(
    technologies_catalog_df[
        [
            "key",
            "name",
            "domain",
            "current_level",
            "l1_cost_spice_eq",
            "l1_effective_time_s",
            "per_level_bonus_pct",
            "prerequisites",
            "strategic_effect",
        ]
    ].to_string(index=False)
)

### 6.3 Spaceships Parameters (`s1`–`s16`) — Costs, Build Times, Combat, Propulsion, Cargo & Orbital Energy (`ships_df`)

This component builds **`ships_df`** (`15` standard SpaceInvasion Classic ship classes indexed via `SHIP_SLUG_BY_KEY`), computing both **base Wiki parameters** and **active-empire effective parameters**:

- **Unit Costs & Ranking Value**: `pig_iron`, `metal`, `kryptonite`, `spice`, `cost_spice_eq`, and `points_value` (`cost_sum / 1,000`).
- **Shipyard Build Time & Throughput**: `base_build_time_s`, `effective_build_time_s = base_build_time_s / (max(1, b9) * 2^b17 * speedup_factor)`, and `units_per_hour_effective`.
- **Zero-Rapidfire (`1:1`) Combat Stats**: Base and tech-boosted `hull` (`+10%/lvl r10`), `shield` (`+10%/lvl r9`), and `attack` (`+10%/lvl r8`), plus `attack_per_spice_eq` and `hull_per_spice_eq`.
- **Propulsion, Logistics & Orbital Energy**: Governing `engine` (`Rocket -> r4`, `Diffusion -> r5`, `Warp -> r6`), `speed_base`, `speed_effective` (`+10%/lvl`), `cargo`, `fuel`, `cargo_per_spice_eq`, orbital `energy_output_effective` (`s9` *Solar cells*, `0` fields used), and tech-tree `prerequisites`.

In [ ]:
_SHIP_ROLE_BY_KEY: dict[str, str] = {
    "s1": "cargo_transport",
    "s2": "cargo_transport",
    "s3": "armed_heavy_hauler",
    "s4": "fodder_interceptor",
    "s5": "early_raider",
    "s6": "cruiser_raider",
    "s7": "fast_strike_cruiser",
    "s8": "colonizer",
    "s9": "orbital_power_satellite",
    "s10": "espionage_probe",
    "s11": "debris_recycler",
    "s13": "capital_warship",
    "s14": "first_strike_bomber",
    "s15": "heavy_destroyer",
    "s16": "supercapital_starbase",
}

_ENGINE_TECH_KEY_MAP: dict[str, str] = {
    "rocket": "r4",
    "diffusion": "r5",
    "warp": "r6",
}

_atk_mult = 1.0 + 0.10 * float(GAME_PARAM_CONTEXT["r8_weapons_tech"])
_shd_mult = 1.0 + 0.10 * float(GAME_PARAM_CONTEXT["r9_shield_tech"])
_hul_mult = 1.0 + 0.10 * float(GAME_PARAM_CONTEXT["r10_defense_systems"])
_eng_mult = 1.0 + 0.01 * float(GAME_PARAM_CONTEXT["r2_energy_tech"])
_planet_temp_c = float(GAME_PARAM_CONTEXT["planet_temperature_c"])

_ship_rows: list[dict[str, Any]] = []
for _skey in sorted(SHIP_SLUG_BY_KEY.keys(), key=lambda k: int(k[1:])):
    _sid = int(_skey[1:])
    _slug = SHIP_SLUG_BY_KEY[_skey]
    _sinfo = game_data.SHIPS[_slug]
    _cost = dict(_sinfo.get("base_cost") or {})
    for _rk in RESOURCE_KEYS:
        _cost.setdefault(_rk, 0)
    _cost_sp = float(game_data.spice_equivalent(_cost))
    _pts_val = float(sum(int(_cost[k]) for k in RESOURCE_KEYS)) / float(POINTS_PER_RESOURCE_DIVISOR)
    _base_bt = float(_sinfo.get("base_build_time_s") or 0.0)
    _eff_bt = _base_bt / SHIPYARD_TIME_DIVISOR
    _units_hr = (3600.0 / _eff_bt) if _eff_bt > 0.0 else 0.0

    _st = _sinfo.get("stats") or {}
    _hull_b = int(_st.get("hull") or 0)
    _shld_b = int(_st.get("shield") or 0)
    _atk_b = int(_st.get("attack") or 0)
    _spd_b = int(_st.get("speed") or 0)
    _cargo = int(_st.get("cargo") or 0)
    _fuel = int(_st.get("fuel") or 0)
    _engine_raw = str(_st.get("engine") or ("Stationary" if _skey == "s9" else "Unknown"))

    _eng_tech_key = "none"
    for _prefix, _tk in _ENGINE_TECH_KEY_MAP.items():
        if _prefix in _engine_raw.lower():
            _eng_tech_key = _tk
            break
    _eng_lvl = int(
        GAME_PARAM_CONTEXT.get(
            {
                "r4": "r4_rocket_engine",
                "r5": "r5_diffusion_engine",
                "r6": "r6_warp_engine",
            }.get(_eng_tech_key, ""),
            0,
        )
    )
    _spd_eff = float(_spd_b) * (1.0 + 0.10 * float(_eng_lvl))

    _solar_base = solar_cell_energy_per_unit(_planet_temp_c) if _skey == "s9" else 0.0
    _solar_eff = _solar_base * _eng_mult

    _ship_rows.append(
        {
            "ship_id": _sid,
            "key": _skey,
            "slug": _slug,
            "name": str(_sinfo["name"]),
            "role": _SHIP_ROLE_BY_KEY.get(_skey, "fleet"),
            "pig_iron": int(_cost["pig_iron"]),
            "metal": int(_cost["metal"]),
            "kryptonite": int(_cost["kryptonite"]),
            "spice": int(_cost["spice"]),
            "cost_spice_eq": round(_cost_sp, 2),
            "points_value": round(_pts_val, 3),
            "base_build_time_s": round(_base_bt, 2),
            "effective_build_time_s": round(_eff_bt, 2),
            "effective_build_time_text": fmt_duration(_eff_bt),
            "units_per_hour_effective": round(_units_hr, 2),
            "hull_base": _hull_b,
            "shield_base": _shld_b,
            "attack_base": _atk_b,
            "hull_effective": round(float(_hull_b) * _hul_mult, 1),
            "shield_effective": round(float(_shld_b) * _shd_mult, 1),
            "attack_effective": round(float(_atk_b) * _atk_mult, 1),
            "engine": _engine_raw,
            "engine_tech_key": _eng_tech_key,
            "speed_base": _spd_b,
            "speed_effective": round(_spd_eff, 1),
            "cargo": _cargo,
            "fuel": _fuel,
            "energy_output_base": round(_solar_base, 2),
            "energy_output_effective": round(_solar_eff, 2),
            "fields_used": 0,
            "cargo_per_spice_eq": round(float(_cargo) / _cost_sp, 3) if _cost_sp > 0.0 else 0.0,
            "attack_per_spice_eq": round((float(_atk_b) * _atk_mult) / _cost_sp, 4) if _cost_sp > 0.0 else 0.0,
            "hull_per_spice_eq": round((float(_hull_b) * _hul_mult) / _cost_sp, 2) if _cost_sp > 0.0 else 0.0,
            "prerequisites": format_prerequisites(_sinfo.get("requirements") or []),
        }
    )

ships_df = pd.DataFrame(_ship_rows)

print(f"Built ships_df ({ships_df.shape[0]} rows x {ships_df.shape[1]} cols)")
print("\n[ships_df — All 15 Spaceships]")
print(
    ships_df[
        [
            "key",
            "name",
            "role",
            "cost_spice_eq",
            "effective_build_time_text",
            "units_per_hour_effective",
            "attack_effective",
            "shield_effective",
            "hull_effective",
            "speed_effective",
            "cargo",
            "fuel",
            "prerequisites",
        ]
    ].to_string(index=False)
)

### 6.4 Planetary Defenses (`d1`–`d11`) & Master Parameter Registry (`defenses_df`, `GAME_PARAMS_TABLES`)

Finally, this component builds **`defenses_df`** (`11` planetary turrets and shield domes indexed via `DEFENSE_SLUG_BY_KEY`, including unit resource costs, `cost_spice_eq`, `points_value`, *Weapons factory* build times, base & tech-boosted `hull` / `shield` / `attack`, `can_dig_in` status, cost-efficiency ratios, and `prerequisites`) and registers all **7 `pandas.DataFrame` parameter tables** inside **`GAME_PARAMS_TABLES`** for direct lookup by the Full Planner strategy simulator.